# Bull Reentry WF2020 Post-Revision Gap Diagnostic V1

Purpose: diagnose why the bull/reentry target revision still leaves `wf_2020` failing after the three-fold checkpoint metric replay.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `bull_reentry_three_fold_checkpoint_metric_replay_v1`; reference `soxlbacktest_qlib_buysell_reference`.
- Role: `diagnose`.
- Target window: `wf_2020`; required context from all three walk-forward windows remains visible.
- Changed layer: model-selection/training-objective feasibility, not actor training.
- Acceptance criteria for this diagnostic: identify whether the remaining `wf_2020` blocker is target coverage, BC fit, checkpoint drift, or gate feasibility; no actor promotion.
- Closest success: `wf_2022` and `wf_2023_2024` pass in the bull/reentry replay.
- Closest failure: pre-revision `cap10_wf2020_failed_fold_objective_audit_v1` identified under-participation and requested bull/reentry target revision.
- Not a duplicate: this notebook starts after the bull/reentry revision and measures the post-revision marginal effect and feasibility of the still-failing `wf_2020` gate.


In [1]:
from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError('could not locate repo root')

EXPERIMENT = 'bull_reentry_wf2020_post_revision_gap_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

OLD_REPLAY = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_three_fold_checkpoint_metric_replay_v2'
NEW_REPLAY = ROOT / 'runtime' / 'model_selection' / 'card' / 'bull_reentry_three_fold_checkpoint_metric_replay_v1'
OLD_AUDIT = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_wf2020_failed_fold_objective_audit_v1'
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'fold_local_bull_reentry_participation_target_v1'
SOXLBT_WF = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1'
SOXLBT_SCORE = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_reference_model_score_v1'

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
REQUIRED_WINDOWS = ['wf_2020', 'wf_2022', 'wf_2023_2024']


In [2]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def load_summary(card: Path) -> dict[str, Any]:
    payload = json.loads((card / 'summary.json').read_text(encoding='utf-8'))
    return payload.get('summary') if isinstance(payload.get('summary'), dict) else payload


def pct_bool(value: Any) -> bool:
    return bool(value) if not pd.isna(value) else False

required_paths = {
    'old_replay_summary': OLD_REPLAY / 'summary.json',
    'new_replay_summary': NEW_REPLAY / 'summary.json',
    'old_replay_candidates': OLD_REPLAY / 'checkpoint_candidate_metrics.csv',
    'new_replay_candidates': NEW_REPLAY / 'checkpoint_candidate_metrics.csv',
    'old_audit_summary': OLD_AUDIT / 'summary.json',
    'old_soxlbt_stage_metrics': OLD_AUDIT / 'soxlbacktest_wf2020_stage_metrics.csv',
    'target_position_audit': TARGET_CARD / 'target_position_audit.csv',
    'target_lift_audit': TARGET_CARD / 'target_lift_audit.csv',
    'new_target_contract': TARGET_CARD / 'bull_reentry_three_fold_target_contract.csv',
    'soxlbt_summary': SOXLBT_WF / 'summary.json',
    'soxlbt_score_selection': SOXLBT_SCORE / 'selection.json',
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'missing required inputs: {missing}')

old_summary = load_summary(OLD_REPLAY)
new_summary = load_summary(NEW_REPLAY)
old_audit_summary = load_summary(OLD_AUDIT)
soxlbt_summary = load_summary(SOXLBT_WF)
soxlbt_score_selection = json.loads(required_paths['soxlbt_score_selection'].read_text(encoding='utf-8'))
old_candidates = pd.read_csv(required_paths['old_replay_candidates'])
new_candidates = pd.read_csv(required_paths['new_replay_candidates'])
soxlbt_stage = pd.read_csv(required_paths['old_soxlbt_stage_metrics'])
target_position_audit = pd.read_csv(required_paths['target_position_audit'])
target_lift_audit = pd.read_csv(required_paths['target_lift_audit'])
target_contract = pd.read_csv(required_paths['new_target_contract'], parse_dates=['datetime'])

print('old decision', old_summary.get('decision'))
print('new decision', new_summary.get('decision'))
print('new failed folds', new_summary.get('failed_folds'))
print('target contract rows', len(target_contract))


old decision three_fold_checkpoint_metric_replay_partial_pass_no_promotion
new decision three_fold_checkpoint_metric_replay_partial_pass_no_promotion
new failed folds ['wf_2020']
target contract rows 3731


In [3]:
rows = []
old_by_fold = old_summary.get('selected_by_fold', {})
new_by_fold = new_summary.get('selected_by_fold', {})
for fold in REQUIRED_WINDOWS:
    old = old_by_fold.get(fold, {})
    new = new_by_fold.get(fold, {})
    rows.append({
        'fold': fold,
        'old_selected_candidate': old.get('selected_candidate'),
        'new_selected_candidate': new.get('selected_candidate'),
        'old_relative_return_pp': old.get('selected_relative_return'),
        'new_relative_return_pp': new.get('selected_relative_return'),
        'relative_delta_pp': None if old.get('selected_relative_return') is None or new.get('selected_relative_return') is None else new.get('selected_relative_return') - old.get('selected_relative_return'),
        'old_max_dd_pct': old.get('selected_max_dd'),
        'new_max_dd_pct': new.get('selected_max_dd'),
        'max_dd_delta_pp': None if old.get('selected_max_dd') is None or new.get('selected_max_dd') is None else new.get('selected_max_dd') - old.get('selected_max_dd'),
        'old_gate': old.get('selected_candidate_gate'),
        'new_gate': new.get('selected_candidate_gate'),
        'post_revision_gap_to_relative_gate_pp': None if new.get('selected_relative_return') is None else TARGET_RELATIVE_RETURN_PP - new.get('selected_relative_return'),
        'post_revision_gap_to_dd_gate_pp': None if new.get('selected_max_dd') is None else max(0.0, new.get('selected_max_dd') - TARGET_MAX_DD_PCT),
    })
pre_post = pd.DataFrame(rows)
pre_post.to_csv(CARD / 'pre_post_replay_delta.csv', index=False)
display(pre_post)


,fold,old_selected_candidate,new_selected_candidate,old_relative_return_pp,new_relative_return_pp,relative_delta_pp,old_max_dd_pct,new_max_dd_pct,max_dd_delta_pp,old_gate,new_gate,post_revision_gap_to_relative_gate_pp,post_revision_gap_to_dd_gate_pp
0,wf_2020,bc_initialization,bc_initialization,-292.753021,-277.735388,15.017634,14.984701,15.489488,0.504787,False,False,307.735388,0.0
1,wf_2022,bc_initialization,bc_initialization,65.864820,65.864820,0.000000,18.802284,18.802284,0.000000,True,True,-35.864820,0.0
2,wf_2023_2024,ppo_retained_iter_4,ppo_retained_iter_4,35.566229,35.566229,0.000000,8.955769,8.955769,0.000000,True,True,-5.566229,0.0


In [4]:
wf2020_new = new_candidates[new_candidates['fold'].eq('wf_2020')].copy()
wf2020_old = old_candidates[old_candidates['fold'].eq('wf_2020')].copy()

# The replay target gate is relative to SOXL buy-and-hold for the same WF test.
selected_new = new_by_fold['wf_2020']
wf2020_buyhold_return = float(wf2020_new['test_buy_and_hold_return'].dropna().iloc[0])
required_wf2020_return = wf2020_buyhold_return + TARGET_RELATIVE_RETURN_PP
selected_return = float(selected_new['selected_relative_return']) + wf2020_buyhold_return

reference_rows = []
for _, row in wf2020_new.iterrows():
    reference_rows.append({
        'source': 'new_replay_candidate',
        'candidate': row['candidate'],
        'return_pct': row['test_return'],
        'max_dd_pct': row['test_max_dd'],
        'avg_position': row['test_avg_pos'],
        'relative_return_pp': row['test_relative_return'],
        'passes_dd_lt20': row['test_max_dd'] < TARGET_MAX_DD_PCT,
        'passes_wf2020_return_gate': row['test_return'] >= required_wf2020_return,
        'passes_wf2020_relative_gate': row['test_relative_return'] >= TARGET_RELATIVE_RETURN_PP,
    })
for _, row in soxlbt_stage[soxlbt_stage['stage'].eq('full_year')].iterrows():
    reference_rows.append({
        'source': 'soxlbacktest_reference_full_year',
        'candidate': row['candidate'],
        'return_pct': row['strategy_return_pct'],
        'max_dd_pct': row['strategy_max_dd_pct'],
        'avg_position': row['avg_position'],
        'relative_return_pp': row['strategy_return_pct'] - wf2020_buyhold_return,
        'passes_dd_lt20': row['strategy_max_dd_pct'] < TARGET_MAX_DD_PCT,
        'passes_wf2020_return_gate': row['strategy_return_pct'] >= required_wf2020_return,
        'passes_wf2020_relative_gate': (row['strategy_return_pct'] - wf2020_buyhold_return) >= TARGET_RELATIVE_RETURN_PP,
    })
reference_rows.append({
    'source': 'gate_threshold',
    'candidate': 'required_wf2020_return_for_relative_plus_30pp',
    'return_pct': required_wf2020_return,
    'max_dd_pct': TARGET_MAX_DD_PCT,
    'avg_position': np.nan,
    'relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'passes_dd_lt20': True,
    'passes_wf2020_return_gate': True,
    'passes_wf2020_relative_gate': True,
})
feasibility = pd.DataFrame(reference_rows)
feasibility['return_gap_to_wf2020_gate_pp'] = required_wf2020_return - feasibility['return_pct']
feasibility['maxdd_gap_to_wf2020_gate_pp'] = (feasibility['max_dd_pct'] - TARGET_MAX_DD_PCT).clip(lower=0)
feasibility['joint_gate_pass'] = feasibility['passes_dd_lt20'].astype(bool) & feasibility['passes_wf2020_relative_gate'].astype(bool)
feasibility.to_csv(CARD / 'wf2020_feasibility_reference_matrix.csv', index=False)
display(feasibility.sort_values(['source', 'return_pct'], ascending=[True, False]))


,source,candidate,return_pct,max_dd_pct,avg_position,relative_return_pp,passes_dd_lt20,passes_wf2020_return_gate,passes_wf2020_relative_gate,return_gap_to_wf2020_gate_pp,maxdd_gap_to_wf2020_gate_pp,joint_gate_pass
8,gate_threshold,required_wf2020_return_for_relative_plus_30pp,363.333347,20.000000,NaN,30.000000,True,True,True,0.000000,0.000000,True
0,new_replay_candidate,bc_initialization,55.597960,15.489488,0.406077,-277.735388,True,False,False,307.735388,0.000000,False
1,new_replay_candidate,ppo_retained_iter_0_bc_equivalent,55.597960,15.489488,0.406077,-277.735388,True,False,False,307.735388,0.000000,False
2,new_replay_candidate,ppo_retained_iter_4,50.224828,15.475617,0.270306,-283.108519,True,False,False,313.108519,0.000000,False
3,new_replay_candidate,ppo_selected_best,50.224828,15.475617,0.270306,-283.108519,True,False,False,313.108519,0.000000,False
4,soxlbacktest_reference_full_year,legacy_default,220.777205,54.275144,0.791023,-112.556142,False,False,False,142.556142,34.275144,False
7,soxlbacktest_reference_full_year,risk_cap_35_15_05,69.604552,17.942350,0.310320,-263.728796,True,False,False,293.728796,0.000000,False
5,soxlbacktest_reference_full_year,wf_pass_ytd18_15_15_02,29.156603,12.785207,0.171011,-304.176744,True,False,False,334.176744,0.000000,False
6,soxlbacktest_reference_full_year,floor_10_03_01,14.651448,3.888779,0.075362,-318.681900,True,False,False,348.681900,0.000000,False


In [5]:
source_contract = target_position_audit[target_position_audit['contract'].eq('source_contract')].iloc[0]
revision = target_position_audit[target_position_audit['contract'].eq('bull_reentry_revision')].iloc[0]
three_fold = target_position_audit[target_position_audit['contract'].eq('bull_reentry_three_fold_revision')].iloc[0]
wf2020_delta = pre_post[pre_post['fold'].eq('wf_2020')].iloc[0]

target_effect = pd.DataFrame([
    {
        'measure': 'wf2020_rows_ge_70pct',
        'source_value': source_contract['rows_ge_70pct'],
        'revision_value': revision['rows_ge_70pct'],
        'delta': revision['rows_ge_70pct'] - source_contract['rows_ge_70pct'],
    },
    {
        'measure': 'wf2020_share_ge_70pct',
        'source_value': source_contract['share_ge_70pct'],
        'revision_value': revision['share_ge_70pct'],
        'delta': revision['share_ge_70pct'] - source_contract['share_ge_70pct'],
    },
    {
        'measure': 'wf2020_mean_risk_target_position',
        'source_value': source_contract['mean_risk_target_position'],
        'revision_value': revision['mean_risk_target_position'],
        'delta': revision['mean_risk_target_position'] - source_contract['mean_risk_target_position'],
    },
    {
        'measure': 'wf2020_selected_relative_return_pp',
        'source_value': wf2020_delta['old_relative_return_pp'],
        'revision_value': wf2020_delta['new_relative_return_pp'],
        'delta': wf2020_delta['relative_delta_pp'],
    },
    {
        'measure': 'wf2020_selected_max_dd_pct',
        'source_value': wf2020_delta['old_max_dd_pct'],
        'revision_value': wf2020_delta['new_max_dd_pct'],
        'delta': wf2020_delta['max_dd_delta_pp'],
    },
])
extra_ge70 = float(revision['rows_ge_70pct'] - source_contract['rows_ge_70pct'])
relative_delta = float(wf2020_delta['relative_delta_pp'])
target_effect['relative_delta_per_added_ge70_row_pp'] = np.where(
    target_effect['measure'].eq('wf2020_selected_relative_return_pp') & (extra_ge70 > 0),
    relative_delta / extra_ge70,
    np.nan,
)
target_effect.to_csv(CARD / 'target_revision_marginal_effect.csv', index=False)
display(target_effect)

target_lift_audit.to_csv(CARD / 'target_lift_audit_copy.csv', index=False)
display(target_lift_audit)


,measure,source_value,revision_value,delta,relative_delta_per_added_ge70_row_pp
0,wf2020_rows_ge_70pct,21.000000,140.000000,119.000000,NaN
1,wf2020_share_ge_70pct,0.008502,0.056680,0.048178,NaN
2,wf2020_mean_risk_target_position,0.312066,0.330183,0.018118,NaN
3,wf2020_selected_relative_return_pp,-292.753021,-277.735388,15.017634,0.126199
4,wf2020_selected_max_dd_pct,14.984701,15.489488,0.504787,NaN


,fold,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,wf_2020_train,2470,149,781,30,149,149,0.386948,0.386948,0.6498,0.6498,True,True,risk,1.000000e-09


In [6]:
# Identify whether the target expansion reached the actual 2020 evaluation window.
contract_time = target_contract.copy()
contract_time['year'] = contract_time['datetime'].dt.year
contract_window = contract_time.groupby(['fold', 'year']).agg(
    rows=('datetime', 'size'),
    ge70_rows=('risk_target_position', lambda s: int((s.astype(float) >= 0.70).sum())),
    bull_reentry_rows=('bull_reentry_target_state', lambda s: int(s.fillna(False).astype(bool).sum()) if s.dtype != object else int(s.astype(str).str.lower().eq('true').sum())),
).reset_index()
contract_window.to_csv(CARD / 'target_contract_year_coverage.csv', index=False)
display(contract_window.tail(20))


,fold,year,rows,ge70_rows,bull_reentry_rows
0,wf_2020_train,2010,206,30,11
1,wf_2020_train,2011,252,1,0
2,wf_2020_train,2012,250,1,0
3,wf_2020_train,2013,252,0,0
4,wf_2020_train,2014,252,33,33
5,wf_2020_train,2015,252,0,0
6,wf_2020_train,2016,252,0,0
7,wf_2020_train,2017,251,0,0
8,wf_2020_train,2018,251,31,31
9,wf_2020_train,2019,252,44,44


In [7]:
selected_feasibility = feasibility[feasibility['candidate'].eq(str(selected_new['selected_candidate']))]
if selected_feasibility.empty:
    selected_feasibility = feasibility[feasibility['source'].eq('new_replay_candidate')].head(1)
selected_feasibility_row = selected_feasibility.iloc[0]
legacy_default = feasibility[feasibility['candidate'].eq('legacy_default')].iloc[0]
risk_cap = feasibility[feasibility['candidate'].eq('risk_cap_35_15_05')].iloc[0]
wf_pass_floor = feasibility[feasibility['candidate'].eq('wf_pass_ytd18_15_15_02')].iloc[0]

joint_pass_count = int(feasibility[feasibility['source'].ne('gate_threshold')]['joint_gate_pass'].sum())
post_revision_relative_improvement = float(pre_post.loc[pre_post['fold'].eq('wf_2020'), 'relative_delta_pp'].iloc[0])
post_revision_relative_gap = float(pre_post.loc[pre_post['fold'].eq('wf_2020'), 'post_revision_gap_to_relative_gate_pp'].iloc[0])
post_revision_dd = float(pre_post.loc[pre_post['fold'].eq('wf_2020'), 'new_max_dd_pct'].iloc[0])

feasibility_blocker = joint_pass_count == 0
if feasibility_blocker:
    decision = 'wf2020_post_revision_gate_feasibility_blocker_no_promotion'
    next_mechanism = 'calibrate_wf2020_retention_gate_or_run_reference_feasibility_screen_before_more_bull_reentry_training'
else:
    decision = 'wf2020_post_revision_target_or_training_gap_no_promotion'
    next_mechanism = 'build_second_pass_bull_reentry_target_with_eval_like_state_coverage_then_rerun_metric_replay'

summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'bull_reentry_three_fold_checkpoint_metric_replay_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'changed_layer': ['model_selection', 'training_objective', 'retention_gate_feasibility'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'selected_by_fold': new_summary.get('selected_by_fold', {}),
    'failed_folds': new_summary.get('failed_folds', []),
    'missing_folds': new_summary.get('missing_folds', []),
    'post_revision_wf2020': {
        'selected_candidate': selected_new.get('selected_candidate'),
        'selected_relative_return_pp': selected_new.get('selected_relative_return'),
        'selected_max_dd_pct': selected_new.get('selected_max_dd'),
        'relative_improvement_vs_pre_revision_pp': post_revision_relative_improvement,
        'relative_gap_to_gate_pp': post_revision_relative_gap,
        'max_dd_pct': post_revision_dd,
    },
    'wf2020_gate_requirements': {
        'buyhold_return_pct': wf2020_buyhold_return,
        'required_return_pct_for_relative_plus_30pp': required_wf2020_return,
        'max_drawdown_pct': TARGET_MAX_DD_PCT,
    },
    'reference_feasibility': {
        'joint_gate_pass_count_excluding_threshold': joint_pass_count,
        'legacy_default_return_pct': float(legacy_default['return_pct']),
        'legacy_default_max_dd_pct': float(legacy_default['max_dd_pct']),
        'risk_cap_35_15_05_return_pct': float(risk_cap['return_pct']),
        'risk_cap_35_15_05_max_dd_pct': float(risk_cap['max_dd_pct']),
        'wf_pass_ytd18_15_15_02_return_pct': float(wf_pass_floor['return_pct']),
        'wf_pass_ytd18_15_15_02_max_dd_pct': float(wf_pass_floor['max_dd_pct']),
        'gate_feasibility_blocker': feasibility_blocker,
    },
    'target_revision_effect': {
        'source_rows_ge_70pct': int(source_contract['rows_ge_70pct']),
        'revision_rows_ge_70pct': int(revision['rows_ge_70pct']),
        'added_rows_ge_70pct': int(revision['rows_ge_70pct'] - source_contract['rows_ge_70pct']),
        'mean_target_position_delta': float(revision['mean_risk_target_position'] - source_contract['mean_risk_target_position']),
        'wf2020_relative_delta_pp': post_revision_relative_improvement,
        'wf2020_max_dd_delta_pp': float(pre_post.loc[pre_post['fold'].eq('wf_2020'), 'max_dd_delta_pp'].iloc[0]),
    },
    'source_decisions': {
        'old_objective_audit': old_audit_summary.get('decision'),
        'old_metric_replay': old_summary.get('decision'),
        'new_metric_replay': new_summary.get('decision'),
        'soxlbacktest_qlib_baseline': soxlbt_summary.get('decision'),
    },
    'next_mechanism': next_mechanism,
    'next_training_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': 'No actor is trained or retained by this diagnostic. It flags the remaining wf_2020 gap and checks whether the current fold gate is feasible against available references.',
    'sidecars': [
        'pre_post_replay_delta.csv',
        'wf2020_feasibility_reference_matrix.csv',
        'target_revision_marginal_effect.csv',
        'target_lift_audit_copy.csv',
        'target_contract_year_coverage.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
    'conclusion': [
        f"Bull/reentry revision improved wf_2020 relative return by {post_revision_relative_improvement:.2f}pp, but the fold remains {post_revision_relative_gap:.2f}pp short of the +30pp relative gate.",
        f"The wf_2020 gate implies required return {required_wf2020_return:.2f}% because SOXL buy-and-hold is {wf2020_buyhold_return:.2f}% in the replay metrics.",
        f"No available reference row satisfies both MaxDD < {TARGET_MAX_DD_PCT:.0f}% and return >= {required_wf2020_return:.2f}%: legacy_default return {legacy_default['return_pct']:.2f}% with MaxDD {legacy_default['max_dd_pct']:.2f}%; risk_cap_35_15_05 return {risk_cap['return_pct']:.2f}% with MaxDD {risk_cap['max_dd_pct']:.2f}%.",
        'Do not start another full retained training run from bull/reentry target tweaks alone until the wf_2020 gate is calibrated or a reference-feasible target is demonstrated.',
    ],
}
selection = dict(summary)
selection['selection_scope'] = 'post_revision_wf2020_gap_diagnostic'
selection['upstream_cards'] = [
    str(OLD_AUDIT.relative_to(ROOT)),
    str(TARGET_CARD.relative_to(ROOT)),
    str(NEW_REPLAY.relative_to(ROOT)),
    str(SOXLBT_WF.relative_to(ROOT)),
]

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

conclusion_md = '\n'.join([
    '# Bull Reentry WF2020 Post-Revision Gap Diagnostic',
    '',
    f"- Decision: `{decision}`.",
    '- Promotion: blocked; no actor is promoted or retained by this diagnostic.',
    f"- Post-revision wf_2020: selected `{selected_new.get('selected_candidate')}` relative `{selected_new.get('selected_relative_return'):.2f}pp`, MaxDD `{selected_new.get('selected_max_dd'):.2f}%`.",
    f"- Bull/reentry target effect: added `{int(revision['rows_ge_70pct'] - source_contract['rows_ge_70pct'])}` high-position target rows and improved relative return by `{post_revision_relative_improvement:.2f}pp`, but the remaining relative gap is `{post_revision_relative_gap:.2f}pp`.",
    f"- Feasibility check: wf_2020 buy-hold is `{wf2020_buyhold_return:.2f}%`, so the current relative gate requires `{required_wf2020_return:.2f}%` return with MaxDD `<20%`.",
    f"- Reference contrast: SOXLBacktest `legacy_default` return `{legacy_default['return_pct']:.2f}%` / MaxDD `{legacy_default['max_dd_pct']:.2f}%`; `risk_cap_35_15_05` return `{risk_cap['return_pct']:.2f}%` / MaxDD `{risk_cap['max_dd_pct']:.2f}%`.",
    f"- Next action: `{next_mechanism}`.",
    '',
    'Sidecars: `pre_post_replay_delta.csv`, `wf2020_feasibility_reference_matrix.csv`, `target_revision_marginal_effect.csv`, `target_contract_year_coverage.csv`, `selection.json`, `summary.json`.',
    '',
])
(CARD / 'conclusion.md').write_text(conclusion_md, encoding='utf-8')

display(Markdown('\n'.join(f'- {item}' for item in summary['conclusion'])))


- Bull/reentry revision improved wf_2020 relative return by 15.02pp, but the fold remains 307.74pp short of the +30pp relative gate.
- The wf_2020 gate implies required return 363.33% because SOXL buy-and-hold is 333.33% in the replay metrics.
- No available reference row satisfies both MaxDD < 20% and return >= 363.33%: legacy_default return 220.78% with MaxDD 54.28%; risk_cap_35_15_05 return 69.60% with MaxDD 17.94%.
- Do not start another full retained training run from bull/reentry target tweaks alone until the wf_2020 gate is calibrated or a reference-feasible target is demonstrated.